# Đây là notebook dùng để giảm kích thước dữ liệu phù hợp cho bài tập lớn cuối kỳ môn máy học

## Load data gốc

In [ ]:
import pandas as pd

df = pd.read_csv('PS_20174392719_1491204439457_log.csv')

fraud_df = df[df["isFraud"] == 1]
non_fraud_df = df[df["isFraud"] == 0]

print("Fraud:", len(fraud_df))
print("Non-fraud:", len(non_fraud_df))


Fraud: 8213
Non-fraud: 6354407


## Downsample non-fraud có stratify theo type

In [5]:
def stratified_downsample_nonfraud(
    non_fraud_df,
    frac=0.05,
    strat_col="type",
    random_state=42
):
    sampled_parts = []

    for t, g in non_fraud_df.groupby(strat_col):
        sampled = g.sample(
            frac=frac,
            random_state=random_state
        )
        sampled_parts.append(sampled)

    return pd.concat(sampled_parts)


In [6]:
non_fraud_sampled = stratified_downsample_nonfraud(
    non_fraud_df,
    frac=0.05
)

## Ghép lại + shuffle nhẹ

In [7]:
df_cut = pd.concat([fraud_df, non_fraud_sampled])
df_cut = df_cut.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_cut["isFraud"].value_counts(normalize=True))

isFraud
0    0.974802
1    0.025198
Name: proportion, dtype: float64


## Giữ cấu trúc thời gian cho train / test

In [8]:
split_step = df_cut["step"].quantile(0.8)

train_df = df_cut[df_cut["step"] <= split_step]
test_df  = df_cut[df_cut["step"] > split_step]

print("Train fraud rate:", train_df["isFraud"].mean())
print("Test fraud rate:", test_df["isFraud"].mean())

Train fraud rate: 0.015188565524367143
Test fraud rate: 0.06567727758089725


## Checklist tự kiểm tra (rất quan trọng!)

### Sau khi cắt, bắt buộc check:

In [ ]:
# 1. Fraud ratio
df_cut["isFraud"].mean()

np.float64(0.025198353040799672)

In [11]:
# 2. Type distribution
df_cut.groupby("type")["isFraud"].mean()

type
CASH_IN     0.000000
CASH_OUT    0.035549
DEBIT       0.000000
PAYMENT     0.000000
TRANSFER    0.134161
Name: isFraud, dtype: float64

In [12]:
# 3. Time coverage
df_cut["step"].min(), df_cut["step"].max()

(np.int64(1), np.int64(743))

# Lưu dữ liệu mới tạo thành dataset mới (dùng trong bài tập lớn)

In [14]:
df_cut.to_csv('C://Users//hi//Downloads//fraud_detection_project//data//raw//Synthetic_Financial_Datasets_For_Fraud_Detection',index=False)